In [ ]:
!pip install -q anthropic pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 23.1 MB/s eta 0:00:00


## conf

In [ ]:
import getpass
import io
import json
import os
import time
from pathlib import Path

import anthropic
import pandas as pd

MODEL = "claude-opus-5"
MAX_TOKENS = 4096
N_ROWS = 1
POLL_SECONDS = 30

CSV_PATH = "analyzingClaudeOpus5TokenConsumption.csv"
RESULTS_PATH = Path("analyzingClaudeOpus5TokenConsumptionResults.csv")
BATCH_ID_PATH = Path("claude_batch_id.txt")

CODE_COLUMNS = {
    "human_code": "human",
    "chatgpt_code": "chatgpt",
    "dsc_code": "deepseek",
    "qwen_code": "qwen",
}

PROMPT = """You are given an existing code implementation and a programming task.

<task>
{task}
</task>

<code>
{code}
</code>

Complete the task. Return a final implementation that fulfills the task."""

In [ ]:
def get_api_key() -> str:
    try:
        from google.colab import userdata
        return userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        return os.environ.get("ANTHROPIC_API_KEY") or getpass.getpass("Anthropic API key: ")


client = anthropic.Anthropic(api_key=get_api_key())

## load data

In [ ]:
def load_dataframe(n_rows=None) -> pd.DataFrame:
    try:
        from google.colab import files
        uploaded = files.upload()
        df = pd.read_csv(io.BytesIO(next(iter(uploaded.values()))))
    except ImportError:
        df = pd.read_csv(CSV_PATH)
    return df.head(n_rows).reset_index(drop=True) if n_rows else df


df = load_dataframe(N_ROWS)
print(f"{len(df)} rows loaded")
df.head()

Saving analyzingClaudeOpus5TokenConsumption.csv to analyzingClaudeOpus5TokenConsumption.csv
1 rows loaded


,hm_index,docstring,docstring_additional,human_code,chatgpt_code,dsc_code,qwen_code
0,gj235730,"Reads a value that can be a map, list, string,...",Handle nested values recursively and preserve ...,private OptionKindAndValue readKindAndValue() ...,public Object readValue(Object value) {\n /...,import com.google.gson.Gson;\nimport com.googl...,private OptionKindAndValue readKindAndValue() ...


## build requests

In [ ]:
def build_runs(df: pd.DataFrame) -> dict[str, dict]:
    """Return {custom_id: metadata} for every non-empty code cell."""
    runs = {}
    for _, row in df.iterrows():
        for column, origin in CODE_COLUMNS.items():
            code_text = row[column]
            if isinstance(code_text, str) and code_text.strip():
                runs[f"run_{len(runs)}"] = {
                    "hm_index": row["hm_index"],
                    "code_origin": origin,
                    "task": row["docstring_additional"],
                    "code": code_text,
                }
    return runs


def to_batch_request(custom_id: str, run: dict) -> dict:
    return {
        "custom_id": custom_id,
        "params": {
            "model": MODEL,
            "max_tokens": MAX_TOKENS,
            "messages": [{
                "role": "user",
                "content": PROMPT.format(task=run["task"], code=run["code"]),
            }],
        },
    }


runs = build_runs(df)
batch_requests = [to_batch_request(cid, run) for cid, run in runs.items()]
print(f"{len(batch_requests)} requests prepared")

4 requests prepared


In [ ]:
batch_requests

[{'custom_id': 'run_0',
  'params': {'model': 'claude-opus-5',
   'max_tokens': 4096,
   'messages': [{'role': 'user',
     'content': 'You are given an existing code implementation and a programming task.\n\n<task>\nHandle nested values recursively and preserve each value’s original type, including empty collections and enum members.\n</task>\n\n<code>\nprivate OptionKindAndValue readKindAndValue() {\n    char peeked = peekChar();\n    switch (peeked) {\n      case \'{\':\n        return OptionKindAndValue.of(OptionElement.Kind.MAP, readMap(\'{\', \'}\', \':\'));\n      case \'[\':\n        return OptionKindAndValue.of(OptionElement.Kind.LIST, readList());\n      case \'"\':\n        return OptionKindAndValue.of(OptionElement.Kind.STRING, readString());\n      default:\n        if (Character.isDigit(peeked) || peeked == \'-\') {\n          return OptionKindAndValue.of(OptionElement.Kind.NUMBER, readWord());\n        }\n        String word = readWord();\n        switch (word) {\n      

## submit batch

In [ ]:
if BATCH_ID_PATH.exists():
    print(f"Found saved batch ID in {BATCH_ID_PATH}; delete the file to submit a new batch.")
    batch_id = BATCH_ID_PATH.read_text().strip()
else:
    batch = client.messages.batches.create(requests=batch_requests)
    batch_id = batch.id
    BATCH_ID_PATH.write_text(batch_id)
    print(f"Batch created: {batch_id}")

Found saved batch ID in claude_batch_id.txt; delete the file to submit a new batch.


## poll until finished

In [ ]:
def wait_for_batch(batch_id: str, poll_seconds: int = POLL_SECONDS):
    while True:
        batch = client.messages.batches.retrieve(batch_id)
        c = batch.request_counts
        print(f"{batch.processing_status}: {c.succeeded} succeeded, {c.errored} errored, "
              f"{c.canceled} canceled, {c.expired} expired, {c.processing} processing")
        if batch.processing_status == "ended":
            return batch
        time.sleep(poll_seconds)


wait_for_batch(batch_id)

ended: 4 succeeded, 0 errored, 0 canceled, 0 expired, 0 processing


MessageBatch(id='msgbatch_01LqSzyqtuSE3ie24srievf9', archived_at=None, cancel_initiated_at=None, created_at=datetime.datetime(2026, 10, 21, 14, 55, 36, 695176, tzinfo=datetime.timezone.utc), ended_at=datetime.datetime(2026, 10, 21, 16, 13, 15, 147868, tzinfo=TzInfo(0)), expires_at=datetime.datetime(2026, 10, 22, 14, 55, 36, 695176, tzinfo=datetime.timezone.utc), processing_status='ended', request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=4), results_url='https://api.anthropic.com/v1/messages/batches/msgbatch_01LqSzyqtuSE3ie24srievf9/results', type='message_batch')

## collect results

In [ ]:
def collect_results(batch_id: str, runs: dict[str, dict]) -> pd.DataFrame:
    rows, failures = [], []
    for item in client.messages.batches.results(batch_id):
        run = runs[item.custom_id]
        if item.result.type != "succeeded":
            failures.append((item.custom_id, item.result.type))
            continue

        message = item.result.message
        usage = message.usage
        rows.append({
            "hm_index": run["hm_index"],
            "code_origin": run["code_origin"],
            "input_tokens": usage.input_tokens,
            "output_tokens": usage.output_tokens,
            "total_tokens": usage.input_tokens + usage.output_tokens,
            "claude_output": "".join(b.text for b in message.content if b.type == "text"),
        })

    print(f"{len(rows)} succeeded, {len(failures)} failed")
    for custom_id, reason in failures:
        print(f"  {custom_id}: {reason}")

    return (pd.DataFrame(rows)
              .sort_values(["hm_index", "code_origin"])
              .reset_index(drop=True))


final = collect_results(batch_id, runs)
final.to_csv(RESULTS_PATH, index=False)
final.head(10)

4 succeeded, 0 failed


,hm_index,code_origin,input_tokens,output_tokens,total_tokens,claude_output
0,gj235730,chatgpt,494,4096,4590,## Implementation\n\nThe key ideas:\n\n* **Rec...
1,gj235730,deepseek,572,4096,4668,## Overview\n\nThe original implementation had...
2,gj235730,human,470,4096,4566,
3,gj235730,qwen,210,4096,4306,## Overview\n\n`readKindAndValue()` is the sin...


## summary

In [ ]:
try:
    from google.colab import files
    files.download(str(RESULTS_PATH))
except ImportError:
    pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>